# AI verklaring



# Wat gedaan met feedback

- Niet direct beginnen met complexe patronen, lichtere varianten proberen
- Veel meer argumenten plaatsen


# CRISP-DM Iteratie 1

## Business Understanding

Studenten vinden het onderwijs- en examenreglement (OER) moeilijk toegankelijk en bestuderen het daardoor onvoldoende. Het doel is een eenvoudige Nederlandstalige chatbot die vragen over het OER beantwoordt. De chatbot mag geen nieuwe feiten verzinnen: als het antwoord niet in de opgehaalde bron staat, moet dat duidelijk worden aangegeven.

### KPI's
- Minimaal 80% van de antwoorden is correct.
- Minimaal 4 van de 5 opgehaalde fragmenten zijn relevant.
- Antwoorden bevatten een bronpagina.
- De chatbot verzint geen informatie.
- Een antwoord verschijnt binnen 60 seconden.

### Kritische succesfactoren (KSF's)
- De OER wordt correct ingelezen en gechunked.
- Het model antwoordt alleen op basis van de context.
- De bronverwijzing is duidelijk.
- De chatbot werkt stabiel tijdens de demo.



## Data Understanding

De kennisbron is het OER PDF. Dit document bevat de opleidingsspecifieke regels voor studenten.

In deze opdracht is er geen sprake van een dataset, het toegang hebben tot specifiek genoemd document (OER) is al prima. 

We gaan er vanuit dat deze OER document inhoudelijk goed is en aan de vereisten voldoet.

## Data Preparation

In deze fase gaan we data: chunken, embedden en opslaan en een juiste start model kiezen.


Eerst de benodigde pdf lezer installeren:

```python
%pip install pypdf chromadb keras-nlp
```



In [2]:
#hier
from sentence_transformers import SentenceTransformer, util

# 1. Laad een voorgetraind embedding-model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Documenten uit de OER en een vraag van een student
documenten = [
    "Je schrijft je uit voor een vak via Studielink.",
    "Het herkansingsbeleid en het aantal toetskansen staan in de OER.",
    "De bibliotheek is open van 8 tot 22 uur.",
]
vraag = "Hoeveel kansen heb ik om een tentamen te halen?"

# 3. Zet zowel de documenten als de vraag om naar vectoren
doc_vectoren = model.encode(documenten)
vraag_vector = model.encode(vraag)

# 4. Bereken de cosine-similarity tussen de vraag en elk document
scores = util.cos_sim(vraag_vector, doc_vectoren)
print(scores)

/Users/irshad/miniconda3/envs/AI_2526_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1110 13:44:20.412000 41884 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9020.01it/s]


tensor([[0.5316, 0.5050, 0.3716]])


We testen eerst of het model al goed genoeg semantisch linken kan leggen. Maar dat blijkt dus niet. 0.53 is het hoogste voorspelde waarde (die ook nog eens fout ligt) we zouden eigenlijk de middelste waarde verwachten maar die heeft 0.50. Nog lager dan verwacht.



In [3]:
# We testen nu een ander model dat wel geschikt is voor soortgelijke cosine-similarities. 

from sentence_transformers import SentenceTransformer, util

# 1. Laad een multilingual embedding-model dat ook Nederlands ondersteunt
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# 2. Gebruik precies dezelfde documenten en vraag als hierboven
documenten = [
    "Je schrijft je uit voor een vak via Studielink.",
    "Het herkansingsbeleid en het aantal toetskansen staan in de OER.",
    "De bibliotheek is open van 8 tot 22 uur.",
]
vraag = "Hoeveel kansen heb ik om een tentamen te halen?"

# 3. Zet documenten en vraag om naar vectoren
doc_vectoren = model.encode(documenten)
vraag_vector = model.encode(vraag)

# 4. Bereken de cosine-similarity
scores = util.cos_sim(vraag_vector, doc_vectoren)
print(scores)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11130.37it/s]


tensor([[0.3495, 0.4366, 0.2295]])


We zien hier dat de resultaat nog niet heel doorslag gevend ligt, maar dat hij nu wel de hoogste waarde op de juiste manier voorspelt. Maar nog niet helemaal zoals we dat gewilt hadden. We gaan in ieder geval verder met deze model. 

We gaan nu de data chunken, embedden en opslaan. 

In [4]:
# 1. Lees de PDF en maak tekstchunks
from pathlib import Path
import re

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

pdf_path = Path("Opleidingsdeel OER Bachelor ICT voltijd Almere 2026-2027.pdf")
reader = PdfReader(str(pdf_path))
if reader.is_encrypted:
    reader.decrypt("")

chunks = []
for page_number, page in enumerate(reader.pages, start=1):
    text = re.sub(r"\s+", " ", page.extract_text() or "").strip()
    words = text.split()
    for start in range(0, len(words), 150):
        chunk = " ".join(words[start:start + 180])
        if len(chunk) >= 80:
            chunks.append({"text": chunk, "page": page_number})
        if start + 180 >= len(words):
            break

print(f"1. {len(chunks)} chunks gemaakt")

# 2. Embed alle chunks met het gekozen Nederlandstalige model
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(texts)
print(f"2. {len(embeddings)} embeddings gemaakt")

# 3. Sla de chunks, embeddings en paginanummers op in ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection(
    "oer_minimal_embeddings_cosine", metadata={"hnsw:space": "cosine"}
)
collection.upsert(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"page": chunk["page"]} for chunk in chunks],
)
print(f"3. {collection.count()} chunks opgeslagen in ChromaDB")

# 4. Test één vraag en toon het beste gevonden resultaat
vraag = "Hoeveel toetskansen heb ik?"
vraag_embedding = embedding_model.encode([vraag])[0].tolist()
resultaat = collection.query(query_embeddings=[vraag_embedding], n_results=1)
print(f"\nTestvraag: {vraag}")
print(f"Pagina: {resultaat['metadatas'][0][0]['page']}")
print(f"Resultaat: {resultaat['documents'][0][0][:300]}...")
print(f"Cosine-similarity: {1 - resultaat['distances'][0][0]:.4f}")


1. 77 chunks gemaakt


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23375.43it/s]


2. 77 embeddings gemaakt
3. 77 chunks opgeslagen in ChromaDB

Testvraag: Hoeveel toetskansen heb ik?
Pagina: 29
Resultaat: 4 • Toetsweek: 28 juni – 2 juli 2027 5.2 Extra kans propedeuse toetsen / versnellen • Schriftelijke toetsen: 9 – 14 aug 2027 • Assessments/producten: 16 – 29 aug 2027 Deze periode is bedoeld voor: • Studenten met 40–59 studiepunten in het 1e jaar van inschrijving in de propedeuse fase (3e kans). Bij...
Cosine-similarity: 0.6205


We hebben hier de vraag omgezet naar embedding/vector, chromaDB gebruikt die vectoren om te vergelijken met alle pdf-chunks, de teksten worden opgehaald en de bijbehorende paginanummer wordt getoond.

ChromaDB kan inmiddels al de relevante vectoren bij elkaar zetten. Zoals paginanummer, resultaat etc. Ik heb nu de LLM nodig die we in voorgaande hoofdstuk bespraken om er een menselijk antwoord op terug te geven. De cosine similarity is ook vrij hoog 0.62

We gaan nu dus verder met RAG (Retrieve, Augmenteren, Genereren) in Modeling. We hebben nu 77 chunks, 77 embeddings en 77 chromadb items

# Modeling


In de vorige stap zagen we dat paraphrase-multilingual betere waardes gaf. Maar dit model kan geen tekst genereren. Het is een embeddingmodel. Voor het genereren van een antwoord gebruiken we Phi-3 model.

Dus paraphrase-multilingual > embedding en retrieval > Phi-3 > Antwoord geven

Middels de Phi-3 zag ik dat de laptop crashte door te veel RAM gebruik. Na wat onderzoek heb ik zelf een LLM gehost: de qwen model met 4B parameters. 

Dus nu ziet het er als volgt uit: Paraphrase: embedding en retrieval > Qwen: antwoord geven 

In [14]:
# 1. Retrieve: zoek meerdere relevante chunks bij de vraag.
vraag = "Hoeveel toetskansen heb ik?"
vraag_embedding = embedding_model.encode([vraag])[0].tolist()
resultaat = collection.query(
    query_embeddings=[vraag_embedding],
    n_results=5,
)

bronnen = [
    {"tekst": tekst, "pagina": metadata["page"]}
    for tekst, metadata in zip(
        resultaat["documents"][0], resultaat["metadatas"][0]
    )
]
print(f"1. Retrieve klaar: {len(bronnen)} relevante chunks gevonden.")
for bron, afstand in zip(bronnen, resultaat["distances"][0]):
    print(
        f"Pagina {bron['pagina']} - cosine-similarity: {1 - afstand:.4f}"
    )

# 2. Augmenteren: voeg de gevonden chunks toe aan de prompt.
context = "\n\n".join(
    f"[Pagina {bron['pagina']}] {bron['tekst']}" for bron in bronnen
)
prompt = f"""Je bent een nauwkeurige Nederlandstalige OER-assistent.
Beantwoord uitsluitend de vraag en gebruik alleen de gegeven context.
Geef een kort en concreet antwoord in maximaal vier zinnen.
Neem voorwaarden, uitzonderingen en doelgroepen uit de context mee.
Verzin niets en trek geen conclusies die niet letterlijk of duidelijk uit de context volgen.
Als de context het antwoord niet bevat, zeg dan: 'Dit kan ik niet vinden in het OER.'
Vermeld aan het einde de gebruikte bronpagina's.

Context:
{context}

Vraag: {vraag}"""
print("2. Augmenteren klaar: context aan prompt toegevoegd.")

# 3. Genereren: laat Qwen een antwoord maken via Ollama.
# Ollama gebruikt een compact GGUF-model en daardoor veel minder geheugen.
import json
from urllib.request import Request, urlopen

ollama_request = Request(
    "http://localhost:11434/api/chat",
    data=json.dumps({
        "model": "qwen2.5:3b",
        "messages": [
            {"role": "system", "content": "Je bent een nauwkeurige OER-assistent. Geef geen informatie die niet in de context staat."},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "options": {"temperature": 0.1},
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urlopen(ollama_request, timeout=300) as response:
    antwoord = json.loads(response.read().decode("utf-8"))["message"]["content"].strip()
print("3. Genereren klaar:\n")
print(antwoord)

1. Retrieve klaar: 3 relevante chunks gevonden.
Pagina 29 - cosine-similarity: 0.6205
Pagina 28 - cosine-similarity: 0.5648
Pagina 25 - cosine-similarity: 0.5212
2. Augmenteren klaar: context aan prompt toegevoegd.
3. Genereren klaar:

De toetskansen zijn voor studenten met 40-59 studiepunten in het 1e jaar van inschrijving in de propedeuse fase. Bij een Ad-opleiding gaat het over de 60 ec van het 1e jaar.


# Evaluation



We zien dat het model nog niet adequaat antwoord kan geven. Zo wijk het antwoord vaak af. We zullen wat optimimalisaties in het RAG toevoegen alvorens we gaan fine-tunen. 


# CRISP-DM Iteratie 2



We gaan wat optimalisaties toevoegen. in eerste instantie op retrieval:

- We halen drie tot vijf relevante chunks op.
- We behouden het paginanummer als bronverwijzing.
- We controleren of de belangrijkste informatie in de opgehaalde context staat.
- We gebruiken het multilingual embeddingmodel voor Nederlandse vragen en OER-tekst.

Hiernaast gaan we de prompt aanscherpen. Zodat Qwen een kort en duidelijk antwoord geeft



In [24]:
# 1. Retrieve: haal vier relevante chunks op
vraag = "Wat zijn de voorwaarde voor afstuderen?"
vraag_embedding = embedding_model.encode([vraag])[0].tolist()

resultaat = collection.query(
    query_embeddings=[vraag_embedding],
    n_results=4,
)

bronnen = [
    {"tekst": tekst, "pagina": metadata["page"]}
    for tekst, metadata in zip(
        resultaat["documents"][0],
        resultaat["metadatas"][0],
    )
]

print(f"Retrieve klaar: {len(bronnen)} chunks gevonden.")
for bron, afstand in zip(bronnen, resultaat["distances"][0]):
    print(
        f"Pagina {bron['pagina']} | "
        f"cosine-similarity: {1 - afstand:.4f}"
    )

# 2. Augmenteren: voeg de bronnen toe aan de prompt
context = "\n\n".join(
    f"[Pagina {bron['pagina']}] {bron['tekst']}"
    for bron in bronnen
)

prompt = f"""Beantwoord de vraag over het OER in duidelijk Nederlands.

Regels:
- Gebruik uitsluitend de informatie uit de context.
- Geef maximaal vier zinnen.
- Neem relevante voorwaarden en uitzonderingen mee.
- Verzin niets en trek geen conclusies die niet uit de context volgen.
- Als de context het antwoord niet bevat, zeg dan exact:
  "Dit kan ik niet vinden in het OER."
- Vermeld aan het einde de gebruikte bronpagina's.

Context:
{context}

Vraag: {vraag}
"""

# 3. Genereren met Qwen via Ollama
ollama_request = Request(
    "http://localhost:11434/api/chat",
    data=json.dumps({
        "model": "qwen2.5:3b",
        "messages": [
            {
                "role": "system",
                "content": (
                    "Je bent een nauwkeurige Nederlandstalige OER-assistent. "
                    "Gebruik uitsluitend de verstrekte context. "
                    "Verzin nooit informatie. "
                    "Volg alle antwoordregels uit de gebruikersprompt."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "options": {"temperature": 0.1},
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)

with urlopen(ollama_request, timeout=300) as response:
    antwoord = json.loads(
        response.read().decode("utf-8")
    )["message"]["content"].strip()

print("\nAntwoord:\n")
print(antwoord)

Retrieve klaar: 4 chunks gevonden.
Pagina 14 | cosine-similarity: 0.7433
Pagina 25 | cosine-similarity: 0.7125
Pagina 19 | cosine-similarity: 0.7080
Pagina 17 | cosine-similarity: 0.7017

Antwoord:

De studenten tot en met cohort 2023 kunnen afstuderen zonder vermelding van een afstudeerrichting. Ze moeten echter de eisen uit Tabel 8 voldoen. De student moet ook een verzoek indienen bij de examencommissie voorafgaand aan het afstuderen.


Met het bijstellen van de chunks en prompt aanpassen zien we dat het antwoord toch wel iets nauwkeuriger begint te worden. Dat zien we graag terug!

Als we de bepaalde chunks aanpassen per vraag dan zien we dat het antwoord daarin tegen ook beter wordt. Hoe kunnen we dit optimaliseren?


# Deployment

